In [ ]:
import pika
import sys

def callback(ch, method, properties, body):
        print(f"Msgs recieved: {body.decode()}")


host = '149.62.71.186'
credentials = pika.PlainCredentials('martin', 'martin00')
parameters =  pika.ConnectionParameters(host=host, credentials=credentials)
connection = pika.BlockingConnection(parameters)
channel = connection.channel()

channel.exchange_declare(exchange='logs', exchange_type='fanout')
result = channel.queue_declare(queue='', exclusive=True)
queue = result.method.queue
channel.queue_bind(exchange='logs', queue=queue)

channel.basic_consume(queue=queue, on_message_callback=callback, auto_ack=True)

channel.start_consuming()
connection.close()

This code implements a RabbitMQ chatroom client that consumes messages from a fanout exchange and prints them. Here's a breakdown:

**1. Imports:**

- `pika`: Library for interacting with RabbitMQ.
- `sys`: Not directly used in this code, but could be useful for command-line arguments.

**2. Callback Function:**

- `def callback(ch, method, properties, body):`: This defines a callback function named `callback`. This function will be called whenever a message is received on the bound queue.
    - The function arguments (`ch`, `method`, `properties`, `body`) hold information about the received message.
    - Inside the function:
        - `body.decode()`: Decodes the message body from bytes to a string (assuming UTF-8 encoding).
        - `print(f"Msgs recieved: {body.decode()}")`: Prints the decoded message body with a prefix "Msgs recieved: ".

**3. Connection and Channel Setup:**

- Defines connection parameters like host, credentials, and establishes a blocking connection.
- Creates a channel on the connection for interacting with exchanges and queues.

**4. Exchange Declaration:**

- Declares a fanout exchange named "logs" for message routing. Remember, a fanout exchange broadcasts messages to all bound queues.

**5. Temporary Queue Creation and Binding:**

- Creates a temporary, exclusive queue for this client using `channel.queue_declare(queue='', exclusive=True)`.
- Binds the created queue to the "logs" exchange using `channel.queue_bind(exchange='logs', queue=queue)`.

**6. Message Consumption:**

- `channel.basic_consume(queue=queue, on_message_callback=callback, auto_ack=True)`: Sets up message consumption from the queue.
    - `queue`: The queue name (obtained earlier).
    - `on_message_callback`: The callback function to be called for each received message (set to `callback`).
    - `auto_ack=True`: Acknowledges receiving the message automatically after processing (simplifies the code but can lead to data loss if processing fails).

**7. Starting Consumption and Closing Connection:**

- `channel.start_consuming()`: Starts consuming messages from the queue (waits for messages and calls the `callback` function).
- `connection.close()`: Closes the connection to the RabbitMQ server after consumption is finished (program termination).

**Improvements:**

- **Consider Manual Acknowledgement:** While `auto_ack` simplifies the code, using manual acknowledgement with proper error handling ensures messages are not lost if processing fails.
- **Exiting the Chatroom:** Implement a mechanism for the user to exit the chatroom gracefully (e.g., using keyboard interrupt or a specific command).

This code provides a solid foundation for a chatroom message receiver. You can enhance it further with features like user identification and displaying messages differently based on the sender.
